# Beam Feature Extraction Pipeline
Extracts physics-grounded features from raw laser beam intensity CSVs (BP and BD) and writes a single output feature matrix CSV.

### Feature Groups
* **A** — Global intensity statistics (24 features)
* **B** — ISO 11146 beam size & shape (14 features)
* **C** — Gaussian fit residual (10 features)
* **D** — ISO 13694 beam quality (16 features)
* **E** — Spatial symmetry (20 features)
* **F** — Higher-order spatial moments (12 features)
* **G** — Edge & clipping (16 features)
* **H** — Regional zone stats 2×4 grid (48 features)
* **I** — BP×BD cross-features (11 features)

In [100]:
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from scipy.optimize import curve_fit
from skimage.transform import downscale_local_mean

import laserbeamsize as lbs
import beamprofiler as bp_lib

# Suppress runtime warnings from expected mathematical edges (e.g., zero divisions)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Configure logging to print nicely inside the notebook output
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [101]:
import os
from pathlib import Path
import numpy as np

def get_project_root(marker_file: str = "pyproject.toml") -> Path:
    """
    Traverse upward from the current file directory to find the project root 
    indicated by the presence of a specific marker file.
    """
    current_path = Path(os.getcwd()).resolve()
    for path in [current_path] + list(current_path.parents):
        if (path / marker_file).exists():
            return path
    return current_path

# Core paths and environment setup
PROJECT_ROOT = get_project_root()

# Updated raw matrix dimensions based on the true 1448*1929 shape
RAW_SHAPE      = (1448, 1929)

# Downsample target keeping the roughly 3:4 aspect ratio intact
# 1448 // 11 = 131, and 1929 // 11 = 175
TARGET_SHAPE   = (131, 175)      

# Integer block size calculation for skimage downscale_local_mean
# Evaluates exactly to (11, 11) using integer division
DOWNSAMPLE_FACTOR = (
    RAW_SHAPE[0] // TARGET_SHAPE[0],
    RAW_SHAPE[1] // TARGET_SHAPE[1],
)  

MAX_VALUE      = 4095              # 12-bit ADC ceiling
SAT_THRESHOLD  = 4095
BG_PERCENTILE  = 5                 # Robust background estimation percentile
EDGE_FRACTION  = 0.05              # Outermost 5% rows/cols = edge zone
ZONE_ROWS      = 2                 # Coarse 2×4 zone grid
ZONE_COLS      = 4

In [102]:
def extract_session_id(path: Path, beam_type: str) -> str:
    """
    Derive a clean, minimal session ID from filenames like "0_0001.ascii.csv".
    Omits the common '0001' suffix token and double extensions.
    """
    # path.stem converts "0_0001.ascii.csv" -> "0_0001.ascii"
    # splitting by '.' drops '.ascii' -> "0_0001"
    base_stem = path.stem.split('.')[0]
    
    # Split by the first underscore to isolate the index from the rest
    parts = base_stem.split('_', 1)
    if len(parts) > 0:
        return parts[0]  # Returns just "0", "1", "10", etc.
        
    return base_stem

def discover_pairs(bp_dir: Path, bd_dir: Path):
    """Scan bp_dir and bd_dir for CSV files and pair them by session ID."""
    bp_map = {extract_session_id(f, "BP"): f for f in sorted(bp_dir.glob("*.csv"))}
    bd_map = {extract_session_id(f, "BD"): f for f in sorted(bd_dir.glob("*.csv"))}

    paired  = {sid: (bp_map[sid], bd_map[sid]) for sid in bp_map if sid in bd_map}
    solo_bp = {sid: bp_map[sid] for sid in bp_map if sid not in bd_map}
    solo_bd = {sid: bd_map[sid] for sid in bd_map if sid not in bp_map}

    log.info(f"Paired sessions: {len(paired)}")
    if solo_bp:
        log.warning(f"BP files with no matching BD ({len(solo_bp)}): {list(solo_bp)[:5]}")
    if solo_bd:
        log.warning(f"BD files with no matching BP ({len(solo_bd)}): {list(solo_bd)[:5]}")

    return paired, solo_bp, solo_bd

In [103]:
def load_and_preprocess(path: Path):
    """Load a raw beam CSV and return a pre-processed float32 array."""
    meta = {"bg_level": np.nan, "frac_saturated": np.nan, "ok": False}
    try:
        raw = pd.read_csv(path, header=None, sep=None, engine='python').values.astype(np.float32)
    except Exception as e:
        log.error(f"Load failed: {path.name} — {e}")
        return None, meta

    if raw.shape != RAW_SHAPE:
        log.error(f"Shape mismatch: {path.name} has {raw.shape}, expected {RAW_SHAPE}")
        return None, meta

    # CRITICAL FIX: Heal NaNs instantly upon loading!
    # We replace them with 0.0 so they don't corrupt the percentile/min/max math
    if np.any(np.isnan(raw)):
        raw = np.nan_to_num(raw, nan=0.0, posinf=4095.0, neginf=0.0)

    # Diagnostic: Check array boundaries safely now
    raw_max = float(np.max(raw))
    raw_min = float(np.min(raw))
    if raw_max == raw_min:
        log.error(f"Flat raw data in {path.name}! All pixels are exactly {raw_max}. Skipping.")
        return None, meta

    # Saturated pixel count
    n_sat = int(np.sum(raw >= SAT_THRESHOLD))
    meta["frac_saturated"] = n_sat / raw.size
    if meta["frac_saturated"] > 0.01:
        log.warning(f"High saturation {meta['frac_saturated']:.2%}: {path.name}")

    # Background subtraction (Now safely immune to NaN poisoning!)
    bg = float(np.percentile(raw, BG_PERCENTILE))
    meta["bg_level"] = bg
    
    if bg >= raw_max:
        log.error(f"Over-clipping in {path.name}: Background level ({bg}) >= Peak signal ({raw_max}).")
        return None, meta

    raw = np.clip(raw - bg, 0, None)

    # Downsample via block mean
    grid = downscale_local_mean(raw, DOWNSAMPLE_FACTOR).astype(np.float32)

    meta["ok"] = True
    return grid, meta

In [104]:
# ── Group A — Global Intensity Statistics ──────────────────────────────────────
def extract_group_a(grid: np.ndarray, prefix: str) -> dict:
    flat   = grid.ravel()
    mean_  = float(np.mean(flat))
    max_   = float(np.max(flat))
    p10    = float(np.percentile(flat, 10))
    p90    = float(np.percentile(flat, 90))
    p95    = float(np.percentile(flat, 95))
    p99    = float(np.percentile(flat, 99))

    return {
        f"{prefix}_global_mean":     mean_,
        f"{prefix}_global_std":      float(np.std(flat)),
        f"{prefix}_global_max":      max_,
        f"{prefix}_global_p10":      p10,
        f"{prefix}_global_p90":      p90,
        f"{prefix}_global_p99":      p99,
        f"{prefix}_peak_to_mean":    max_ / mean_ if mean_ > 0 else np.nan,
        f"{prefix}_p90_to_mean":     p90 / mean_ if mean_ > 0 else np.nan,
        f"{prefix}_frac_above_p90":  float(np.mean(flat > p90)),
        f"{prefix}_frac_above_p95":  float(np.mean(flat > p95)),
    }

# ── Group B — ISO 11146 Beam Size & Shape ─────────────────────────────────────
def extract_group_b(grid: np.ndarray, prefix: str) -> dict:
    h, w = grid.shape
    
    # 1. Coordinate grids
    ys, xs = np.indices(grid.shape, dtype=np.float64)
    
    # 2. Total intensity (zeroth moment)
    total_intensity = float(np.sum(grid))
    if total_intensity <= 0:
        log.warning(f"Manual ISO failed for {prefix}: Zero or negative total intensity.")
        return {k: np.nan for k in [
            f"{prefix}_iso_centroid_x", f"{prefix}_iso_centroid_y",
            f"{prefix}_iso_d_major", f"{prefix}_iso_d_minor",
            f"{prefix}_iso_ellipticity", f"{prefix}_iso_rotation_deg",
            f"{prefix}_iso_eccentricity",
        ]}
        
    # 3. Centroids (first moments)
    x_bar = float(np.sum(xs * grid) / total_intensity)
    y_bar = float(np.sum(ys * grid) / total_intensity)
    
    # 4. Central variances (second moments)
    sigma_x2 = float(np.sum(((xs - x_bar) ** 2) * grid) / total_intensity)
    sigma_y2 = float(np.sum(((ys - y_bar) ** 2) * grid) / total_intensity)
    sigma_xy = float(np.sum((xs - x_bar) * (ys - y_bar) * grid) / total_intensity)
    
    # 5. Core ISO Math with robust guards
    diff = sigma_x2 - sigma_y2
    sum_sigmas = sigma_x2 + sigma_y2
    discriminant = np.sqrt(diff**2 + 4 * (sigma_xy**2))
    
    # Terms under the radical for diameters
    major_term = (sum_sigmas + discriminant) / 2.0
    minor_term = (sum_sigmas - discriminant) / 2.0
    
    # CRITICAL SECURITY GUARD: Force absolute clipping to prevent imaginary numbers
    if major_term <= 0 or minor_term <= 0:
        log.warning(f"Manual ISO edge case encountered for {prefix}: Non-positive variance terms.")
        # If minor_term is negative due to noise, clip it to 0 or drop to NaN
        minor_term = max(0.0, minor_term)
        if major_term <= 0:
            major_term = np.nan

    # 6. Diameters (ISO 4-sigma definitions)
    d_major = 4.0 * np.sqrt(major_term) if np.isfinite(major_term) else np.nan
    d_minor = 4.0 * np.sqrt(minor_term) if np.isfinite(minor_term) else np.nan
    
    # 7. Angle, Ellipticity, Eccentricity
    phi = 0.5 * np.arctan2(2 * sigma_xy, diff)
    
    if np.isfinite(d_major) and d_major > 0:
        ellipticity = d_minor / d_major
        # Bound the eccentricity calculation to avoid domain errors if d_minor > d_major slightly due to float noise
        ecc_val = 1.0 - (d_minor / d_major) ** 2
        eccentricity = float(np.sqrt(max(0.0, ecc_val)))
    else:
        ellipticity = np.nan
        eccentricity = np.nan

    return {
        f"{prefix}_iso_centroid_x":   float(x_bar / w),
        f"{prefix}_iso_centroid_y":   float(y_bar / h),
        f"{prefix}_iso_d_major":      float(d_major / w) if np.isfinite(d_major) else np.nan,
        f"{prefix}_iso_d_minor":      float(d_minor / h) if np.isfinite(d_minor) else np.nan,
        f"{prefix}_iso_ellipticity":  float(ellipticity),
        f"{prefix}_iso_rotation_deg": float(np.degrees(phi)),
        f"{prefix}_iso_eccentricity": eccentricity,
    }

# ── Group C — Gaussian Fit Residual ───────────────────────────────────────────
def _gaussian_2d(xy, A, x0, y0, sigma_x, sigma_y, offset):
    x, y = xy
    return (offset + A * np.exp(-0.5 * (((x - x0) / sigma_x) ** 2 + ((y - y0) / sigma_y) ** 2))).ravel()

def extract_group_c(grid: np.ndarray, prefix: str) -> dict:
    h, w = grid.shape
    xs = np.arange(w, dtype=np.float32)
    ys = np.arange(h, dtype=np.float32)
    X, Y = np.meshgrid(xs, ys)

    A0, x0_0, y0_0 = float(grid.max()), float(w / 2), float(h / 2)
    sx0, sy0, off0 = float(w / 4), float(h / 4), float(grid.min())

    try:
        popt, _ = curve_fit(_gaussian_2d, (X.ravel(), Y.ravel()), grid.ravel(), p0=[A0, x0_0, y0_0, sx0, sy0, off0], maxfev=10_000)
        A, x0, y0, sigma_x, sigma_y, offset = popt
        fitted = _gaussian_2d((X, Y), *popt).reshape(h, w)
        residual = grid - fitted
        return {
            f"{prefix}_gauss_amplitude":     float(A),
            f"{prefix}_gauss_sigma_x":       float(abs(sigma_x) / w),
            f"{prefix}_gauss_sigma_y":       float(abs(sigma_y) / h),
            f"{prefix}_gauss_residual_rmse": float(np.sqrt(np.mean(residual ** 2))),
            f"{prefix}_gauss_residual_max":  float(np.max(np.abs(residual))),
            f"{prefix}_gauss_fit_ok":        True,
        }
    except RuntimeError:
        log.warning(f"Gaussian fit did not converge for {prefix}")
        return {k: np.nan for k in [f"{prefix}_gauss_amplitude", f"{prefix}_gauss_sigma_x", f"{prefix}_gauss_sigma_y", f"{prefix}_gauss_residual_rmse", f"{prefix}_gauss_residual_max"]} | {f"{prefix}_gauss_fit_ok": False}

# ── Group D — ISO 13694 Beam Quality ──────────────────────────────────────────
def extract_group_d(grid: np.ndarray, prefix: str) -> dict:
    """
    Computes ISO 13694 power distribution parameters using discrete 
    intensity thresholds to evaluate uniformity and edge steepness.
    """
    total_power = float(np.sum(grid))
    if total_power <= 0:
        return {k: np.nan for k in [
            f"{prefix}_iso_flatness_factor", f"{prefix}_iso_beam_uniformity",
            f"{prefix}_iso_plateau_uniformity", f"{prefix}_iso_edge_steepness",
            f"{prefix}_iso_fractional_power", f"{prefix}_iso_clip_area",
            f"{prefix}_iso_beam_aspect_ratio", f"{prefix}_iso_top_hat_factor"
        ]}
        
    # Find the peak power point (Maximum power density E_max)
    E_max = float(np.max(grid))
    
    # Calculate standard ISO clip levels: 10% (outer edge boundary) and 90% (inner plateau boundary)
    clip_10 = 0.10 * E_max
    clip_90 = 0.90 * E_max
    
    # Clip Area (A_clip): Total physical footprint where power >= 10% of peak.
    clip_mask = grid >= clip_10
    A_clip = float(np.sum(clip_mask))
    
    # Flatness Factor: Ratio of mean power within the plateau zone to the peak power.
    # Formula: Mean(E_plateau) / E_max  [Approaches 1.0 for a perfect laser flat-top/top-hat]
    plateau_mask = grid >= clip_90
    E_mean_plateau = float(np.mean(grid[plateau_mask])) if np.any(plateau_mask) else 0.0
    flatness_factor = E_mean_plateau / E_max if E_max > 0 else np.nan
    
    # Top-Hat Factor: Overall structural efficiency parameter.
    # Formula: Mean(E_all_active_pixels) / E_max
    E_mean_total = float(np.mean(grid[grid > 0])) if np.any(grid > 0) else 0.0
    top_hat_factor = E_mean_total / E_max if E_max > 0 else np.nan
    
    # Beam Uniformity: Normalized standard deviation across the active beam area.
    # Formula: Std(E_active) / Mean(E_active) [Lower numbers mean a smoother, cleaner beam]
    beam_uniformity = float(np.std(grid[clip_mask]) / np.mean(grid[clip_mask])) if A_clip > 0 else np.nan
    
    # Plateau Uniformity: Normalized standard deviation restricted purely to the 90% peak zone.
    # Formula: Std(E_plateau) / Mean(E_plateau) [Measures ripples/noise on top of the beam head]
    plateau_uniformity = float(np.std(grid[plateau_mask]) / E_mean_plateau) if np.any(plateau_mask) else np.nan
    
    # Edge Steepness: Measures how fast the laser transitions from nothing to maximum intensity.
    # Formula: 1.0 - (Area_at_90_percent / Area_at_10_percent)
    # A step-cylinder jumps instantly (Area_90 == Area_10), meaning Edge Steepness = 1.0 (perfectly sharp).
    A_90 = float(np.sum(grid >= clip_90))
    edge_steepness = 1.0 - (A_90 / A_clip) if A_clip > 0 else np.nan
    
    # Fractional Power: Ratio of energy contained inside the 10% clip zone compared to total matrix power.
    # Formula: Sum(E_active) / Sum(E_total_matrix)
    fractional_power = float(np.sum(grid[clip_mask])) / total_power
    
    # Aspect Ratio: Spatial geometric symmetry based on the active 10% footprint.
    # Formula: Min(bounding_width, bounding_height) / Max(bounding_width, bounding_height)
    rows, cols = np.where(clip_mask)
    if len(rows) > 0 and len(cols) > 0:
        dx = float(np.max(cols) - np.min(cols) + 1)
        dy = float(np.max(rows) - np.min(rows) + 1)
        aspect_ratio = min(dx, dy) / max(dx, dy) if max(dx, dy) > 0 else np.nan
    else:
        aspect_ratio = np.nan

    return {
        f"{prefix}_iso_flatness_factor":    flatness_factor,
        f"{prefix}_iso_beam_uniformity":    beam_uniformity,
        f"{prefix}_iso_plateau_uniformity": plateau_uniformity,
        f"{prefix}_iso_edge_steepness":     edge_steepness,
        f"{prefix}_iso_fractional_power":   fractional_power,
        f"{prefix}_iso_clip_area":          A_clip / grid.size, # Normalized to total sensor grid size
        f"{prefix}_iso_beam_aspect_ratio":  aspect_ratio,
        f"{prefix}_iso_top_hat_factor":     top_hat_factor,
    }

# ── Group E — Spatial Symmetry ────────────────────────────────────────────────
def extract_group_e(grid: np.ndarray, prefix: str) -> dict:
    h, w = grid.shape
    total = float(grid.sum()) or 1.0

    mid_r, mid_c = h // 2, w // 2
    top, bottom = grid[:mid_r, :].sum(), grid[mid_r:, :].sum()
    left, right = grid[:, :mid_c].sum(), grid[:, mid_c:].sum()

    q1 = grid[:mid_r, :mid_c].sum() / total
    q2 = grid[:mid_r, mid_c:].sum() / total
    q3 = grid[mid_r:, :mid_c].sum() / total
    q4 = grid[mid_r:, mid_c:].sum() / total

    col_profile = grid.sum(axis=0)
    row_profile = grid.sum(axis=1)

    return {
        f"{prefix}_sym_horizontal":   float((left - right) / total),
        f"{prefix}_sym_vertical":     float((top - bottom) / total),
        f"{prefix}_sym_q1":           float(q1),
        f"{prefix}_sym_q2":           float(q2),
        f"{prefix}_sym_q3":           float(q3),
        f"{prefix}_sym_q4":           float(q4),
        f"{prefix}_sym_diag_1":       float((q1 + q4 - q2 - q3)),
        f"{prefix}_sym_diag_2":       float((q1 + q2 - q3 - q4)),
        f"{prefix}_marginal_skew_x":  float(sp_stats.skew(col_profile)),
        f"{prefix}_marginal_skew_y":  float(sp_stats.skew(row_profile)),
    }

# ── Group F — Higher-Order Spatial Moments ────────────────────────────────────
def extract_group_f(grid: np.ndarray, prefix: str) -> dict:
    col_profile = grid.sum(axis=0).astype(np.float64)
    row_profile = grid.sum(axis=1).astype(np.float64)
    xs = np.arange(len(col_profile), dtype=np.float64)
    ys = np.arange(len(row_profile), dtype=np.float64)

    def weighted_median(values, weights):
        weights = np.asarray(weights, dtype=np.float64)
        total = weights.sum()
        if total == 0:
            return np.nan
        cumw = np.cumsum(weights / total)
        return float(values[np.searchsorted(cumw, 0.5)])

    return {
        f"{prefix}_moment_kurtosis_x":    float(sp_stats.kurtosis(col_profile)),
        f"{prefix}_moment_kurtosis_y":    float(sp_stats.kurtosis(row_profile)),
        f"{prefix}_moment_skewness_x":    float(sp_stats.skew(col_profile)),
        f"{prefix}_moment_skewness_y":    float(sp_stats.skew(row_profile)),
        f"{prefix}_moment_energy_x_p50":  weighted_median(xs, col_profile),
        f"{prefix}_moment_energy_y_p50":  weighted_median(ys, row_profile),
    }

# ── Group G — Edge & Clipping ─────────────────────────────────────────────────
def extract_group_g(grid: np.ndarray, prefix: str) -> dict:
    h, w   = grid.shape
    total  = float(grid.sum()) or 1.0
    n_r    = max(1, int(h * EDGE_FRACTION))
    n_c    = max(1, int(w * EDGE_FRACTION))

    top, bottom = float(grid[:n_r, :].sum()), float(grid[-n_r:, :].sum())
    left, right = float(grid[:, :n_c].sum()), float(grid[:, -n_c:].sum())

    corner = float(grid[:n_r, :n_c].sum() + grid[:n_r, -n_c:].sum() + grid[-n_r:, :n_c].sum() + grid[-n_r:, -n_c:].sum())

    return {
        f"{prefix}_edge_top_frac":     top    / total,
        f"{prefix}_edge_bottom_frac":  bottom / total,
        f"{prefix}_edge_left_frac":    left   / total,
        f"{prefix}_edge_right_frac":   right  / total,
        f"{prefix}_edge_total_frac":   (top + bottom + left + right) / total,
        f"{prefix}_edge_asymmetry_h":  (left - right) / total,
        f"{prefix}_edge_asymmetry_v":  (top - bottom) / total,
        f"{prefix}_edge_corner_frac":  corner / total,
    }

# ── Group H — Regional Zone Statistics ────────────────────────────────────────
def extract_group_h(grid: np.ndarray, prefix: str) -> dict:
    h, w = grid.shape
    row_edges = np.linspace(0, h, ZONE_ROWS + 1, dtype=int)
    col_edges = np.linspace(0, w, ZONE_COLS + 1, dtype=int)

    features = {}
    for r in range(ZONE_ROWS):
        for c in range(ZONE_COLS):
            zone = grid[row_edges[r]:row_edges[r + 1], col_edges[c]:col_edges[c + 1]]
            flat = zone.ravel()
            features[f"{prefix}_zone_{r}_{c}_mean"] = float(np.mean(flat))
            features[f"{prefix}_zone_{r}_{c}_std"]  = float(np.std(flat))
            features[f"{prefix}_zone_{r}_{c}_p90"]  = float(np.percentile(flat, 90))
    return features

# ── Group I — BP × BD Cross-Features ──────────────────────────────────────────
def extract_group_i(bp_feats: dict, bd_feats: dict) -> dict:
    safe_sub = lambda a, b: a - b if (np.isfinite(a) and np.isfinite(b)) else np.nan
    safe_div = lambda a, b: a / b if (np.isfinite(a) and np.isfinite(b) and b != 0) else np.nan

    return {
        "cross_centroid_x_delta":   safe_sub(bp_feats.get("BP_iso_centroid_x", np.nan), bd_feats.get("BD_iso_centroid_x", np.nan)),
        "cross_centroid_y_delta":   safe_sub(bp_feats.get("BP_iso_centroid_y", np.nan), bd_feats.get("BD_iso_centroid_y", np.nan)),
        "cross_d_major_ratio":      safe_div(bp_feats.get("BP_iso_d_major", np.nan), bd_feats.get("BD_iso_d_major", np.nan)),
        "cross_d_minor_ratio":      safe_div(bp_feats.get("BP_iso_d_minor", np.nan), bd_feats.get("BD_iso_d_minor", np.nan)),
        "cross_ellipticity_delta":  safe_sub(bp_feats.get("BP_iso_ellipticity", np.nan), bd_feats.get("BD_iso_ellipticity", np.nan)),
        "cross_rotation_delta":     safe_sub(bp_feats.get("BP_iso_rotation_deg", np.nan), bd_feats.get("BD_iso_rotation_deg", np.nan)),
        "cross_peak_ratio":         safe_div(bp_feats.get("BP_global_max", np.nan), bd_feats.get("BD_global_max", np.nan)),
        "cross_uniformity_delta":   safe_sub(bp_feats.get("BP_iso_beam_uniformity", np.nan), bd_feats.get("BD_iso_beam_uniformity", np.nan)),
        "cross_flatness_delta":     safe_sub(bp_feats.get("BP_iso_flatness_factor", np.nan), bd_feats.get("BD_iso_flatness_factor", np.nan)),
        "cross_sym_h_delta":        safe_sub(bp_feats.get("BP_sym_horizontal", np.nan), bd_feats.get("BD_sym_horizontal", np.nan)),
        "cross_edge_total_delta":   safe_sub(bp_feats.get("BP_edge_total_frac", np.nan), bd_feats.get("BD_edge_total_frac", np.nan)),
    }

In [105]:
def extract_single_beam(path: Path, prefix: str) -> tuple[dict | None, dict]:
    """Runs full single-matrix extraction pipeline for either BP or BD."""
    grid, meta = load_and_preprocess(path)
    
    # STEP 1: Guard first! Ensure data was successfully loaded
    if not meta["ok"] or grid is None:
        return None, meta

    # STEP 2: Now it is safe to clean up NaNs / Infs if they exist
    if np.any(np.isnan(grid)) or np.any(np.isinf(grid)):
        grid = np.nan_to_num(grid, nan=0.0, posinf=0.0, neginf=0.0)

    # STEP 3: Ensure the entire matrix wasn't completely wiped out to zero
    if np.max(grid) == 0:
        log.warning(f"Matrix is completely zero after preprocessing for {path.name}")
        meta["ok"] = False
        return None, meta

    feats = {}
    feats.update(extract_group_a(grid, prefix))
    feats.update(extract_group_b(grid, prefix))
    feats.update(extract_group_c(grid, prefix))
    feats.update(extract_group_d(grid, prefix))
    feats.update(extract_group_e(grid, prefix))
    feats.update(extract_group_f(grid, prefix))
    feats.update(extract_group_g(grid, prefix))
    feats.update(extract_group_h(grid, prefix))

    feats[f"{prefix}_bg_level"]       = meta["bg_level"]
    feats[f"{prefix}_frac_saturated"] = meta["frac_saturated"]

    return feats, meta


def process_session(session_id: str, bp_path: Path, bd_path: Path) -> dict:
    """Orchestrates pipeline execution for an isolated cross-beam session."""
    log.info(f"Processing {session_id}")
    row = {"session_id": session_id}

    bp_feats, bp_meta = extract_single_beam(bp_path, "BP")
    bd_feats, bd_meta = extract_single_beam(bd_path, "BD")

    row["BP_extraction_ok"] = bp_meta["ok"]
    row["BD_extraction_ok"] = bd_meta["ok"]

    if bp_feats:
        row.update(bp_feats)
    if bd_feats:
        row.update(bd_feats)

    if bp_feats and bd_feats:
        row.update(extract_group_i(bp_feats, bd_feats))
    else:
        cross_keys = [
            "cross_centroid_x_delta", "cross_centroid_y_delta", "cross_d_major_ratio", 
            "cross_d_minor_ratio", "cross_ellipticity_delta", "cross_rotation_delta", 
            "cross_peak_ratio", "cross_uniformity_delta", "cross_flatness_delta", 
            "cross_sym_h_delta", "cross_edge_total_delta"
        ]
        for k in cross_keys:
            row[k] = np.nan

    return row

In [106]:
def run_extraction(bp_dir: Path, bd_dir: Path, label: str, output: Path):
    """Executes the complete directory scanning, processing, and output compilation."""
    log.info(f"=== Beam Feature Extraction | label={label} ===")
    log.info(f"  BP dir : {bp_dir}")
    log.info(f"  BD dir : {bd_dir}")
    log.info(f"  Output : {output}")

    paired, solo_bp, solo_bd = discover_pairs(bp_dir, bd_dir)

    if not paired:
        log.error("No paired sessions found. Check filename conventions.")
        return

    rows = []
    for session_id, (bp_path, bd_path) in paired.items():
        try:
            row = process_session(session_id, bp_path, bd_path)
            row["label"] = label
            rows.append(row)
        except Exception as e:
            log.error(f"Unhandled error for session {session_id}: {e}", exc_info=True)
            rows.append({"session_id": session_id, "label": label, "BP_extraction_ok": False, "BD_extraction_ok": False})

    df = pd.DataFrame(rows)

    # Reorder columns to place execution status metrics forward
    front = ["session_id", "label", "BP_extraction_ok", "BD_extraction_ok"]
    other = [c for c in df.columns if c not in front]
    df = df[front + other]

    output.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output, index=False)

    n_ok = int(df["BP_extraction_ok"].sum() & df["BD_extraction_ok"].sum())
    log.info(f"Done. {len(df)} sessions processed ({n_ok} fully OK).")
    log.info(f"Feature columns: {len(df.columns) - len(front)}. Output: {output}")

    # Zone-feature allocation share sanity report
    zone_cols = [c for c in df.columns if "_zone_" in c]
    total_feat = len(df.columns) - len(front)
    log.info(f"Zone features: {len(zone_cols)} / {total_feat} = {len(zone_cols)/total_feat:.1%} of total (cap: 50%)")

In [107]:
import os
from pathlib import Path

def get_project_root(marker_file: str = "pyproject.toml") -> Path:
    """
    Traverse upward from the current file directory to find the project root 
    indicated by the presence of a specific marker file.
    """
    # In a notebook, os.getcwd() returns the directory containing the .ipynb file
    current_path = Path(os.getcwd()).resolve()
    
    # Loop through the directory and its parents
    for path in [current_path] + list(current_path.parents):
        if (path / marker_file).exists():
            return path
            
    # Fallback to current directory if marker file is nowhere to be found
    return current_path

# Extract and assign the project root
PROJECT_ROOT = get_project_root()
print(f"Project Root Identified: {PROJECT_ROOT}")

Project Root Identified: C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data


In [108]:
# Hardcode execution directories and parameters here
BP_INPUT_DIR_AMP = Path(f"{PROJECT_ROOT}/share_IBM_training/AFM_BP")
BD_INPUT_DIR_AMP = Path(f"{PROJECT_ROOT}/share_IBM_training/AFM_BD")
SUBSYSTEM_LABEL_AMP = "AMP"
OUTPUT_FILEPATH_AMP = Path(f"{PROJECT_ROOT}/processed/beam_physical_feature_amp.csv")

In [109]:
# Hardcode execution directories and parameters here
BP_INPUT_DIR_OSC = Path(f"{PROJECT_ROOT}/share_IBM_training/OSC_BP")
BD_INPUT_DIR_OSC = Path(f"{PROJECT_ROOT}/share_IBM_training/OSC_BD")
SUBSYSTEM_LABEL_OSC = "OSC"
OUTPUT_FILEPATH_OSC = Path(f"{PROJECT_ROOT}/processed/beam_physical_feature_osc.csv")

In [110]:
import pandas as pd
import numpy as np

# Adjust path to where your file actually sits
test_path = BP_INPUT_DIR_AMP / "0_0001.ascii.csv"

df = pd.read_csv(test_path, header=None, sep=None, engine='python')
print("Parsed Shape:", df.shape)
print("Data Types summary:\n", df.dtypes.value_counts())
print("Raw Array Max:", np.max(df.values))
print("Raw Array Min:", np.min(df.values))
print("First 5x5 corner of the matrix:\n", df.iloc[:5, :5])

Parsed Shape: (1448, 1929)
Data Types summary:
 float64    1929
Name: count, dtype: int64
Raw Array Max: nan
Raw Array Min: nan
First 5x5 corner of the matrix:
            0          1          2          3          4
0  57.500000  62.699999  64.900000  57.400000  66.599998
1  53.299999  53.799999  59.799999  57.000000  50.400000
2  63.799999  65.099998  71.799999  54.900000  64.099998
3  52.900000  62.199999  64.299999  56.500000  69.000000
4  64.000000  53.599998  55.199999  59.599998  64.699999


In [111]:
# Call the engine processing function with predefined parameters
run_extraction(
    bp_dir=BP_INPUT_DIR_AMP,
    bd_dir=BD_INPUT_DIR_AMP,
    label=SUBSYSTEM_LABEL_AMP,
    output=OUTPUT_FILEPATH_AMP
)

16:26:07  INFO      === Beam Feature Extraction | label=AMP ===
16:26:07  INFO        BP dir : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\share_IBM_training\AFM_BP
16:26:07  INFO        BD dir : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\share_IBM_training\AFM_BD
16:26:07  INFO        Output : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\processed\beam_physical_feature_amp.csv
16:26:07  INFO      Paired sessions: 271
16:26:07  INFO      Processing 0
16:26:13  INFO      Processing 100
16:26:20  INFO      Processing 101
16:26:29  INFO      Processing 102
16:26:36  INFO      Processing 103
16:26:44  INFO      Processing 104
16:26:51  INFO      Processing 105
16:26:59  INFO      Processing 106
16:27:06  INFO      Processing 107
16:27:13  INFO      Processing 108
16:27:20  INFO      Processing 109
16:27:27  INFO      Processing 10
16:27:35  INFO      Processing 110
16:27:42  INFO      Processing 111
16:2

In [112]:
# Call the engine processing function with predefined parameters
run_extraction(
    bp_dir=BP_INPUT_DIR_OSC,
    bd_dir=BD_INPUT_DIR_OSC,
    label=SUBSYSTEM_LABEL_OSC,
    output=OUTPUT_FILEPATH_OSC
)

17:00:41  INFO      === Beam Feature Extraction | label=OSC ===
17:00:41  INFO        BP dir : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\share_IBM_training\OSC_BP
17:00:41  INFO        BD dir : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\share_IBM_training\OSC_BD
17:00:41  INFO        Output : C:\Users\BS01493\Projects\SBU Europe\Client\GigaAI\Client Data - v3\data\processed\beam_physical_feature_osc.csv
17:00:41  INFO      Paired sessions: 307
17:00:41  INFO      Processing 0
17:00:49  INFO      Processing 100
17:00:57  INFO      Processing 101
17:01:04  INFO      Processing 102
17:01:15  INFO      Processing 103
17:01:23  INFO      Processing 104
17:01:38  WARNING   Gaussian fit did not converge for BD
17:01:38  INFO      Processing 105
17:01:54  WARNING   Gaussian fit did not converge for BD
17:01:54  INFO      Processing 106
C:\Users\BS01493\AppData\Local\Temp\ipykernel_14224\3617282428.py:109: OptimizeWarning: Covariance 